In [7]:
import pandas as pd
import re

df = pd.read_csv('File 3.csv')
df.shape

(418, 12)

In [8]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [9]:
def fix_name(name):
    cleaned = name.replace('"', '')
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

df['Name'] = df['Name'].apply(fix_name)

In [10]:
str_cols = ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
for c in str_cols:
    df[c] = df[c].astype('string').str.strip()

In [11]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')[0].str.strip()

title_map = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs', 'Lady': 'Rare', 'Countess': 'Rare',
    'Capt': 'Rare', 'Col': 'Rare', 'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare',
    'Rev': 'Rare', 'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
}
df['Title'] = df['Title'].replace(title_map)

In [12]:
df['Age'] = df.groupby(['Title', 'Pclass'])['Age'].transform(lambda s: s.fillna(s.median()))
df['Age'] = df['Age'].fillna(df['Age'].median())

In [13]:
df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda s: s.fillna(s.median()))

In [14]:
df['Deck'] = df['Cabin'].str[0]
df['Deck'] = df['Deck'].fillna('Unknown')
df['HasCabin'] = df['Cabin'].notna().astype(int)
df['Cabin'] = df['Cabin'].fillna('Unknown')

In [15]:
df['PassengerId'] = df['PassengerId'].astype(int)
df['Survived'] = df['Survived'].astype(int)
df['Pclass'] = df['Pclass'].astype(int)
df['SibSp'] = df['SibSp'].astype(int)
df['Parch'] = df['Parch'].astype(int)
df['Age'] = df['Age'].round(2)
df['Fare'] = df['Fare'].round(4)

In [16]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

In [17]:
print(df.duplicated(subset=['PassengerId']).sum())
print(df.isna().sum())
df.head()

0
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
Title          0
Deck           0
HasCabin       0
FamilySize     0
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Deck,HasCabin,FamilySize
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,Unknown,Q,Mr,Unknown,0,1
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,Unknown,S,Mrs,Unknown,0,2
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,Unknown,Q,Mr,Unknown,0,1
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,Unknown,S,Mr,Unknown,0,1
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,Unknown,S,Mrs,Unknown,0,3


In [18]:
df.to_csv('File_3_cleaned.csv', index=False)